# Huấn luyện Phase 1 GAN (Adversarial Training) trên Colab


## 1. Cài đặt môi trường & Kết nối Google Drive


In [ ]:
!pip install -q wandb gdown matplotlib torchaudio tensorboard soundfile pandas tqdm einops pesq pystoi pyyaml torchmetrics

from google.colab import drive
drive.mount('/content/drive')

import os
import sys
from pathlib import Path

# Nếu bạn đang dùng workspace khác, đổi dòng này trước khi chạy tiếp.
WORK_DIR = Path('/content/drive/MyDrive/DeepVQE_Workspace')

os.chdir(WORK_DIR)

SHARED_CHECKPOINTS_DIR = Path('/content/drive/MyDrive/DeepVQE_Workspace2')

print(f'Thư mục làm việc hiện tại: {Path.cwd()}')
print(f'Python executable: {sys.executable}')
print(f'Checkpoint root: {SHARED_CHECKPOINTS_DIR}')

## 2. Clone mã nguồn DeepVQE


In [ ]:
# Repo này có sẵn deepvqe.py, ablation/ và stream/ benchmark scripts.
GIT_REPO = "https://github.com/hoxuanphu/Pdeepvqe.git"
REPO_DIR = "deepvqe"

import os
if not os.path.exists(REPO_DIR):
    !git clone {GIT_REPO} {REPO_DIR}
else:
    print(f"Thư mục {REPO_DIR} đã tồn tại. Đang cập nhật code mới nhất từ GitHub...")
    !cd {REPO_DIR} && git pull origin main

os.chdir(REPO_DIR)
import sys
sys.path.insert(0, str(os.getcwd()))
print(f"Đã vào thư mục mã nguồn: {os.getcwd()}")


## 2.5 Kiểm tra cấu hình model

Cell này kiểm tra nhanh runtime đã nhận đúng cấu hình Baseline trước khi khởi tạo model cho quá trình training GAN.


In [ ]:
from ablation.deepvqe_ablation import get_ablation_config
base_cfg = get_ablation_config('Baseline')
print('Baseline config OK:', base_cfg)


## 3. Tải bộ dữ liệu VoiceBank-DEMAND

Dữ liệu gốc được tải từ các ZIP trên Google Drive. Notebook ưu tiên lấy ZIP đã cache trong Google Drive, sau đó copy sang Local SSD `/content/data` để train nhanh.


In [ ]:
import os
import zipfile
import shutil
from pathlib import Path

# Cache ZIP trên Google Drive để lần sau không phải tải lại.
drive_data_dir = WORK_DIR / 'data' / 'voicebank-demand'
drive_data_dir.mkdir(parents=True, exist_ok=True)

# Local SSD của Colab để giải nén và train nhanh.
data_dir = Path('/content/data/voicebank-demand')
data_dir.mkdir(parents=True, exist_ok=True)

datasets = [
    ('clean_trainset_28spk_wav.zip', 'https://drive.google.com/file/d/1NJr2O4Ik6ueSFlIGSvub8dnFXGTHJ2PG/view?usp=sharing'),
    ('noisy_trainset_28spk_wav.zip', 'https://drive.google.com/file/d/1OqpDIvpVyaTnMbwY1Qt__hfX3X4siMtU/view?usp=sharing'),
    ('clean_testset_wav.zip', 'https://drive.google.com/file/d/1GQc-T1R4FNrhRjTn7AAvAenZTIQEazeH/view?usp=sharing'),
    ('noisy_testset_wav.zip', 'https://drive.google.com/file/d/1rimmCqxXRYRIXZcPkGjQiacr6j1QsMAH/view?usp=sharing'),
]

for filename, url in datasets:
    drive_zip = drive_data_dir / filename
    local_zip = data_dir / filename
    extract_folder = data_dir / filename.replace('.zip', '')

    if drive_zip.exists():
        print(f'Đã tìm thấy {filename} trên Drive cache.')
        if not local_zip.exists() and not extract_folder.exists():
            print(f'Copy {filename} từ Drive sang Local SSD...')
            shutil.copy2(drive_zip, local_zip)
    else:
        print(f'Không thấy {filename} trên Drive cache; tải xuống Local SSD...')
        if not extract_folder.exists():
            if local_zip.exists() and not zipfile.is_zipfile(local_zip):
                print(f'File {local_zip} bị lỗi, đang xóa...')
                local_zip.unlink()
            if not local_zip.exists():
                os.system(f'gdown --fuzzy {url} -O {local_zip}')
            if local_zip.exists() and zipfile.is_zipfile(local_zip):
                print(f'Lưu cache {filename} lên Drive...')
                shutil.copy2(local_zip, drive_zip)

    if not extract_folder.exists():
        if local_zip.exists():
            print(f'Đang giải nén {filename}...')
            with zipfile.ZipFile(local_zip, 'r') as zip_ref:
                zip_ref.extractall(data_dir)
            print(f'Hoàn tất giải nén {filename}.')
        else:
            print(f'Lỗi: Không tìm thấy file {local_zip} để giải nén!')

print('Dữ liệu đã sẵn sàng trên Local SSD của Colab.')


## 4. Xử lý và phân chia Dataset (Tạo file CSV)

Tạo `train.csv`, `valid.csv`, `test.csv` giống baseline v4: train/valid split 90/10 với `random_state=42`.


In [ ]:
import glob
import pandas as pd
from pathlib import Path

data_dir = Path(data_dir)


def create_csv(clean_dir, noisy_dir, output_csv):
    clean_files = sorted(glob.glob(str(Path(clean_dir) / '*.wav')))
    noisy_files = sorted(glob.glob(str(Path(noisy_dir) / '*.wav')))

    assert len(clean_files) == len(noisy_files), f'Số lượng file không khớp! ({len(clean_files)} != {len(noisy_files)})'
    mismatched = [
        (Path(c).name, Path(n).name)
        for c, n in zip(clean_files, noisy_files)
        if Path(c).name != Path(n).name
    ]
    assert not mismatched, f'Tên file clean/noisy không khớp: {mismatched[:5]}'

    rows = []
    for clean, noisy in zip(clean_files, noisy_files):
        filename = Path(clean).name
        rows.append({
            'ID': filename.replace('.wav', ''),
            'clean_wav': str(Path(clean).resolve()),
            'noisy_wav': str(Path(noisy).resolve()),
        })

    df = pd.DataFrame(rows)
    df.to_csv(output_csv, index=False)
    print(f'Đã tạo {output_csv} với {len(df)} mẫu.')


create_csv(data_dir / 'clean_trainset_28spk_wav', data_dir / 'noisy_trainset_28spk_wav', data_dir / 'train_full.csv')
create_csv(data_dir / 'clean_testset_wav', data_dir / 'noisy_testset_wav', data_dir / 'test.csv')

train_full = pd.read_csv(data_dir / 'train_full.csv')
train_full = train_full.sample(frac=1, random_state=42).reset_index(drop=True)
split_idx = int(len(train_full) * 0.90)
train_full.iloc[:split_idx].to_csv(data_dir / 'train.csv', index=False)
train_full.iloc[split_idx:].to_csv(data_dir / 'valid.csv', index=False)

print(f'Train: {split_idx} | Valid: {len(train_full) - split_idx}')
print(f'Test: {len(pd.read_csv(data_dir / "test.csv"))}')
print('Hoàn tất tạo metadata CSV!')


## 5. Cấu hình Hyperparameters

Các giá trị bám baseline v4: seed `1234`, STFT `sqrt_hann`, batch size `8`, optimizer Adam, loss RI/Mag/SI-SNR. Notebook chỉ train `B3b`.


In [ ]:
import json
from pathlib import Path

data_dir = Path(data_dir)
WORK_DIR = Path(WORK_DIR)

CONFIG = {
    'config_id': 'Baseline',
    'seed': 1234,
    'sample_rate': 16000,
    'n_fft': 512,
    'hop_length': 256,
    'win_length': 512,
    'stft_window': 'sqrt_hann',

    'batch_size': 8,
    'valid_batch_size': 4,
    'grad_accum_steps': 1,
    'num_workers': 2,
    'persistent_workers': True,
    'prefetch_factor': 2,
    'progress_update_every': 10,
    'epochs': 80,
    'lr': 1e-3,
    'weight_decay': 0.0,
    'grad_clip': 5.0,

    'lamda_ri': 30.0,
    'lamda_mag': 70.0,
    'compress_factor': 0.3,
    
    # GAN Configs
    'use_gan': True,
    'num_d_scales': 1,
    'lamda_adv': 0.05,
    'lambda_fm': 0.0,

    'train_csv': str(data_dir / 'train.csv'),
    'valid_csv': str(data_dir / 'valid.csv'),
    'test_csv': str(data_dir / 'test.csv'),

    'checkpoint_dir': str(SHARED_CHECKPOINTS_DIR / 'deepvqe_vctk' ),
    'output_dir': str(SHARED_CHECKPOINTS_DIR / 'deepvqe_vctk' ),
    'resume_checkpoint': 'last.pt',
    'resume_existing': False,
    'results_dir': str(WORK_DIR / 'results' / 'ablation'),
    'onnx_dir': str(WORK_DIR / 'onnx_models' / 'ablation'),

    'scheduler_factor': 0.5,
    'scheduler_patience': 5,
    'scheduler_min_lr': 1e-6,
    'early_stopping_patience': 15,
    'early_stopping_min_delta': 0.0,
    'early_stopping_min_epochs': 0,

    'use_amp': True,
    'augment': True,
    'aug_gain_range_db': (-6.0, 6.0),
    'aug_snr_remix_range': (0.0, 20.0),
    'aug_prob': 0.5,

    'valid_random_crop': True,

    'tensorboard': False,
    'use_wandb': False,
    'wandb_project': 'DeepVQE-Phase1-GAN',
    'eval_pesq_every': 5,
    'eval_pesq_samples': 50,
    'log_audio_every': 0,

    'run_training': True,
    'run_smoke_tests': True,
    'run_arch_benchmark': True,
    'run_quality_eval': True,
    'run_onnx_export': False,
}

for key in ['checkpoint_dir', 'output_dir', 'results_dir', 'onnx_dir']:
    Path(CONFIG[key]).mkdir(parents=True, exist_ok=True)

print(json.dumps(CONFIG, indent=2, default=str))



## 6. Dataset & DataLoader (Pure PyTorch)

Cùng kiểu baseline v4: crop đoạn 3 giây để tiết kiệm VRAM, augment chỉ áp dụng cho training set.


In [ ]:
import random
import numpy as np
import torch
import torchaudio
from torch.utils.data import Dataset, DataLoader


def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything(CONFIG['seed'])


def repeat_to_length(wav, length):
    if wav.numel() == 0:
        raise ValueError('Encountered empty waveform')
    if wav.numel() >= length:
        return wav
    repeats = int(np.ceil(length / wav.numel()))
    return wav.repeat(repeats)[:length]


def crop_pair(noisy, clean, segment_len, random_crop=True):
    min_len = min(noisy.shape[0], clean.shape[0])
    noisy = noisy[:min_len]
    clean = clean[:min_len]

    if segment_len is None:
        return noisy, clean
    if min_len <= segment_len:
        return noisy, clean

    if random_crop:
        start = random.randint(0, min_len - segment_len)
    else:
        start = max(0, (min_len - segment_len) // 2)
    return noisy[start:start + segment_len], clean[start:start + segment_len]


class VCTKDemandDataset(Dataset):
    """Dataset cho VCTK-DEMAND: đọc cặp (noisy, clean) từ CSV."""

    def __init__(self, csv_path, sample_rate=16000, segment_len=None, augment_cfg=None, random_crop=True):
        self.df = pd.read_csv(csv_path)
        self.sample_rate = sample_rate
        self.segment_len = segment_len
        self.aug = augment_cfg
        self.random_crop = random_crop

    def __len__(self):
        return len(self.df)

    def _load(self, path):
        wav, sr = torchaudio.load(path)
        wav = wav.float()
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if sr != self.sample_rate:
            wav = torchaudio.functional.resample(wav, sr, self.sample_rate)
        return wav.squeeze(0)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        noisy = self._load(row['noisy_wav'])
        clean = self._load(row['clean_wav'])
        noisy, clean = crop_pair(noisy, clean, self.segment_len, random_crop=self.random_crop)

        if self.aug is not None:
            aug_prob = self.aug.get('aug_prob', 0.5)

            if random.random() < aug_prob:
                lo, hi = self.aug['aug_gain_range_db']
                gain_db = random.uniform(lo, hi)
                gain = 10 ** (gain_db / 20.0)
                noisy = noisy * gain
                clean = clean * gain

            if random.random() < aug_prob:
                noise = noisy - clean
                lo, hi = self.aug['aug_snr_remix_range']
                target_snr = random.uniform(lo, hi)
                clean_rms = clean.pow(2).mean().sqrt().clamp(min=1e-8)
                noise_rms = noise.pow(2).mean().sqrt().clamp(min=1e-8)
                target_noise_rms = clean_rms / (10 ** (target_snr / 20.0))
                noisy = clean + noise * (target_noise_rms / noise_rms)

        return {'noisy': noisy, 'clean': clean, 'id': row['ID']}


def collate_fn(batch):
    max_len = max(item['noisy'].shape[0] for item in batch)
    noisy_batch, clean_batch, ids = [], [], []
    for item in batch:
        noisy, clean = item['noisy'], item['clean']
        pad_len = max_len - noisy.shape[0]
        if pad_len > 0:
            noisy = torch.nn.functional.pad(noisy, (0, pad_len))
            clean = torch.nn.functional.pad(clean, (0, pad_len))
        noisy_batch.append(noisy)
        clean_batch.append(clean)
        ids.append(item['id'])

    return {
        'noisy': torch.stack(noisy_batch),
        'clean': torch.stack(clean_batch),
        'id': ids,
    }


segment_len = int(3.0 * CONFIG['sample_rate'])
aug_cfg = {k: v for k, v in CONFIG.items() if k.startswith('aug_')} if CONFIG.get('augment') else None

train_dataset = VCTKDemandDataset(
    CONFIG['train_csv'],
    CONFIG['sample_rate'],
    segment_len=segment_len,
    augment_cfg=aug_cfg,
    random_crop=True,
)
valid_dataset = VCTKDemandDataset(
    CONFIG['valid_csv'],
    CONFIG['sample_rate'],
    segment_len=segment_len,
    augment_cfg=None,
    random_crop=CONFIG.get('valid_random_crop', True),
)

loader_kwargs = dict(
    num_workers=CONFIG['num_workers'],
    pin_memory=torch.cuda.is_available(),
    collate_fn=collate_fn,
)
if CONFIG['num_workers'] > 0:
    loader_kwargs.update(
        persistent_workers=CONFIG.get('persistent_workers', False),
        prefetch_factor=CONFIG.get('prefetch_factor', 2),
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG['batch_size'],
    shuffle=True,
    drop_last=True,
    **loader_kwargs,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=CONFIG.get('valid_batch_size', CONFIG['batch_size']),
    shuffle=False,
    drop_last=False,
    **loader_kwargs,
)

print(f'Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'Valid: {len(valid_dataset)} samples, {len(valid_loader)} batches')
print(f'Valid crop: {"random như baseline v4" if CONFIG.get("valid_random_crop", True) else "center/fixed"}')
if aug_cfg:
    print(f'Augmentation: ON (gain={aug_cfg["aug_gain_range_db"]}, snr_remix={aug_cfg["aug_snr_remix_range"]}, prob={aug_cfg["aug_prob"]})')
else:
    print('Augmentation: OFF')


## 7. Khởi tạo Model B3b, Optimizer, Scheduler

Khác baseline gốc ở đúng một điểm: model dùng `DeepVQE_Ablation.from_config_id('B3b')`, tức thêm SE frequency skip-gating vào skip-connection.


In [ ]:
import time
import json
from pathlib import Path
from torch.utils.tensorboard import SummaryWriter

from ablation.ablation_config import deep_update, get_train_config, reproducibility_metadata
from ablation.deepvqe_ablation import DeepVQE_Ablation, count_parameters
from ablation.discriminator import Discriminator, MultiScaleDiscriminator

def make_grad_scaler(enabled):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'GradScaler'):
        return torch.amp.GradScaler('cuda', enabled=enabled)
    return torch.cuda.amp.GradScaler(enabled=enabled)

def autocast_context(device, enabled):
    if hasattr(torch, 'amp') and hasattr(torch.amp, 'autocast'):
        return torch.amp.autocast(device.type, enabled=enabled)
    return torch.cuda.amp.autocast(enabled=enabled)

# Build TRAIN_CFG omitting boilerplate mapping
TRAIN_CFG = get_train_config(CONFIG['config_id'])

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model = DeepVQE_Ablation.from_config_id(CONFIG['config_id']).to(device)
total_params = count_parameters(model)
print(f'Generator params: {total_params / 1e6:.2f}M')

if CONFIG['num_d_scales'] > 1:
    model_D = MultiScaleDiscriminator(num_scales=CONFIG['num_d_scales']).to(device)
else:
    model_D = Discriminator().to(device)
print(f'Discriminator params: {count_parameters(model_D) / 1e6:.2f}M')

optimizer = torch.optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'], min_lr=CONFIG['scheduler_min_lr']
)

opt_D = torch.optim.Adam(model_D.parameters(), lr=1e-4, betas=(0.5, 0.999))
scheduler_D = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_D, mode='min', factor=CONFIG['scheduler_factor'],
    patience=CONFIG['scheduler_patience'] + 2, min_lr=1e-7
)

use_amp = CONFIG['use_amp'] and device.type == 'cuda'
scaler = make_grad_scaler(enabled=use_amp)
scaler_D = make_grad_scaler(enabled=use_amp)
print(f'Mixed Precision (AMP): {"ON" if use_amp else "OFF"}')

window = torch.hann_window(CONFIG['win_length']).sqrt().to(device)
output_dir = Path(CONFIG.get('checkpoint_dir', CONFIG['output_dir']))
output_dir.mkdir(parents=True, exist_ok=True)



## 8. Hàm tiện ích: STFT, Loss, Checkpoint, Log, Metrics

Giữ cùng loss baseline v4: compressed RI + compressed magnitude + negative SI-SNR trên waveform gốc.


In [ ]:
import csv
import shutil
import numpy as np

from ablation.discriminator import adversarial_d_loss, adversarial_g_loss, feature_matching_loss

def _run_d(model_D, x, return_features=False):
    if return_features:
        outputs, features = model_D(x, return_features=True)
        if not isinstance(outputs, list):
            return [outputs], [features]
        return outputs, features
    result = model_D(x)
    if not isinstance(result, list):
        return [result]
    return result

def make_stft(wav, n_fft, hop_length, win_length, win):
    spec = torch.stft(wav, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=win, return_complex=True)
    return torch.view_as_real(spec)

def make_istft(spec, n_fft, hop_length, win_length, win, length=None):
    complex_spec = torch.complex(spec[..., 0], spec[..., 1])
    return torch.istft(complex_spec, n_fft=n_fft, hop_length=hop_length, win_length=win_length, window=win, length=length)

def compute_loss_base(model, noisy_wav, clean_wav, cfg, win):
    n_fft = cfg['n_fft']
    hop = cfg['hop_length']
    win_len = cfg['win_length']
    c = cfg['compress_factor']

    noisy_spec = make_stft(noisy_wav, n_fft, hop, win_len, win)

    amp_enabled = cfg.get('use_amp', False) and noisy_wav.device.type == 'cuda'
    with autocast_context(noisy_wav.device, enabled=amp_enabled):
        pred_stft = model(noisy_spec)

    pred_stft = pred_stft.float()
    clean_spec = make_stft(clean_wav, n_fft, hop, win_len, win)

    min_t = min(pred_stft.shape[2], clean_spec.shape[2])
    pred_stft = pred_stft[:, :, :min_t, :]
    true_stft = clean_spec[:, :, :min_t, :]

    pred_real, pred_imag = pred_stft[..., 0], pred_stft[..., 1]
    true_real, true_imag = true_stft[..., 0], true_stft[..., 1]

    pred_mag = torch.sqrt(pred_real**2 + pred_imag**2 + 1e-12)
    true_mag = torch.sqrt(true_real**2 + true_imag**2 + 1e-12)

    pred_real_c = pred_real / (pred_mag**(1 - c))
    pred_imag_c = pred_imag / (pred_mag**(1 - c))
    true_real_c = true_real / (true_mag**(1 - c))
    true_imag_c = true_imag / (true_mag**(1 - c))

    real_loss = torch.mean((pred_real_c - true_real_c)**2)
    imag_loss = torch.mean((pred_imag_c - true_imag_c)**2)
    mag_loss = torch.mean((pred_mag**c - true_mag**c)**2)

    y_pred = make_istft(pred_stft, n_fft, hop, win_len, win, length=clean_wav.shape[-1])
    y_true = clean_wav
    min_wav_len = min(y_pred.shape[-1], y_true.shape[-1])
    y_pred = y_pred[..., :min_wav_len]
    y_true = y_true[..., :min_wav_len]

    eps = 1e-8
    true_energy = torch.sum(torch.square(y_true), dim=-1, keepdim=True)
    y_target = torch.sum(y_true * y_pred, dim=-1, keepdim=True) * y_true / (true_energy + eps)
    target_energy = torch.sum(torch.square(y_target), dim=-1, keepdim=True)
    noise_energy = torch.sum(torch.square(y_pred - y_target), dim=-1, keepdim=True)
    sisnr = -10 * torch.log10((target_energy + eps) / (noise_energy + eps)).mean()

    loss = cfg['lamda_ri'] * (real_loss + imag_loss) + cfg['lamda_mag'] * mag_loss + sisnr
    return loss, {
        'loss': loss.item(),
        'ri_loss': (real_loss + imag_loss).item(),
        'mag_loss': mag_loss.item(),
        'sisnr': sisnr.item(),
    }, pred_stft, clean_spec

def unwrap_model(model):
    return model.module if hasattr(model, 'module') else model

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler=None, model_D=None, opt_D=None, scaler_D=None, scheduler_D=None):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    ckpt = {
        'model': unwrap_model(model).state_dict(),
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict() if scheduler else None,
        'scaler': scaler.state_dict() if scaler else None,
        'epoch': epoch,
        'best_loss': best_loss,
        'bad_epochs': bad_epochs,
        'config': TRAIN_CFG,
    }
    if model_D: ckpt['model_D'] = unwrap_model(model_D).state_dict()
    if opt_D: ckpt['opt_D'] = opt_D.state_dict()
    if scaler_D: ckpt['scaler_D'] = scaler_D.state_dict()
    if scheduler_D: ckpt['scheduler_D'] = scheduler_D.state_dict()
    
    torch.save(ckpt, str(path))

def load_checkpoint(path, model, optimizer=None, scheduler=None, device='cpu', scaler=None, model_D=None, opt_D=None, scaler_D=None, scheduler_D=None):
    ckpt = torch.load(str(path), map_location=device, weights_only=False)
    unwrap_model(model).load_state_dict(ckpt['model'])
    if optimizer and ckpt.get('optimizer'): optimizer.load_state_dict(ckpt['optimizer'])
    if scheduler and ckpt.get('scheduler'): scheduler.load_state_dict(ckpt['scheduler'])
    if scaler and ckpt.get('scaler'): scaler.load_state_dict(ckpt['scaler'])
    
    if model_D and ckpt.get('model_D'): unwrap_model(model_D).load_state_dict(ckpt['model_D'])
    if opt_D and ckpt.get('opt_D'): opt_D.load_state_dict(ckpt['opt_D'])
    if scaler_D and ckpt.get('scaler_D'): scaler_D.load_state_dict(ckpt['scaler_D'])
    if scheduler_D and ckpt.get('scheduler_D'): scheduler_D.load_state_dict(ckpt['scheduler_D'])
    
    return ckpt.get('epoch', 0), ckpt.get('best_loss'), ckpt.get('bad_epochs', 0)

def set_optimizer_lr(optimizer, lr):
    for group in optimizer.param_groups:
        group['lr'] = lr

def append_log(log_path, row_dict):
    log_path = Path(log_path)
    file_exists = log_path.exists() and log_path.stat().st_size > 0
    with open(log_path, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=list(row_dict.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)


def compute_pesq_stoi(*args, **kwargs):
    return {'pesq': 0.0, 'stoi': 0.0}



## 8.5 Khởi động TensorBoard (tùy chọn)


In [ ]:
if CONFIG.get('tensorboard', False):
    ip = get_ipython()
    ip.run_line_magic('load_ext', 'tensorboard')
    ip.run_line_magic('tensorboard', f'--logdir {str(output_dir / "tb_logs")}')
else:
    print('TensorBoard đang tắt trong CONFIG để ưu tiên tốc độ train.')


## 8.6 Đăng nhập WandB (tùy chọn)


In [ ]:
if CONFIG.get('use_wandb', False):
    import os
    import wandb

    wandb_key = os.environ.get('WANDB_API_KEY')
    if not wandb_key:
        try:
            from google.colab import userdata
            wandb_key = userdata.get('WANDB_API_KEY')
        except Exception as exc:
            print(f'Không lấy được WANDB_API_KEY từ Colab Secrets/env: {exc}')

    if wandb_key:
        wandb.login(key=wandb_key, relogin=True)
        print('Đã đăng nhập WandB bằng WANDB_API_KEY.')
    else:
        print('Chưa có WANDB_API_KEY; wandb.init có thể yêu cầu đăng nhập thủ công.')
else:
    print('WandB đang tắt trong CONFIG. Đổi use_wandb=True nếu muốn log lên WandB.')


## 9. Vòng lặp huấn luyện B3b (Training Loop)

B3b mặc định `resume_existing=False` để lần chạy đầu train từ scratch. Nếu Colab session bị ngắt sau khi B3b đã tạo checkpoint hợp lệ, đổi `resume_existing=True` trong CONFIG để tiếp tục từ `B3b_scratch_v1/last.pt`.


In [ ]:
try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

wandb = None
if CONFIG.get('use_wandb', False):
    import wandb as _wandb
    wandb = _wandb
    wandb.init(project=CONFIG.get('wandb_project', 'DeepVQE-Phase1-GAN'), config=CONFIG, resume='allow')

start_epoch = 0
best_loss = None
bad_epochs = 0

resume_path = output_dir / CONFIG.get('resume_checkpoint', 'last.pt')
if CONFIG.get('resume_existing', True) and resume_path.exists():
    start_epoch, best_loss, bad_epochs = load_checkpoint(
        resume_path, model, optimizer, scheduler, device, scaler,
        model_D, opt_D, scaler_D, scheduler_D
    )
    print(f'Resumed from epoch {start_epoch}')

log_path = output_dir / 'train_log.csv'
accum_steps = CONFIG.get('grad_accum_steps', 1)
amp_enabled = CONFIG['use_amp'] and device.type == 'cuda'

if CONFIG.get('run_training', True):
    for epoch in range(start_epoch + 1, CONFIG['epochs'] + 1):
        model.train()
        model_D.train()
        
        train_losses = []
        valid_accum_batches = 0
        t0 = time.time()
        
        optimizer.zero_grad(set_to_none=True)
        opt_D.zero_grad(set_to_none=True)

        pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]', leave=False)
        for batch_idx, batch in enumerate(pbar):
            noisy = batch['noisy'].to(device, non_blocking=True)
            clean = batch['clean'].to(device, non_blocking=True)

            loss_G_base, parts, est_spec, tgt_spec = compute_loss_base(model, noisy, clean, CONFIG, window)
            
            est_mag = torch.sqrt(est_spec[..., 0]**2 + est_spec[..., 1]**2 + 1e-12).float()
            tgt_mag = torch.sqrt(tgt_spec[..., 0]**2 + tgt_spec[..., 1]**2 + 1e-12).float()

            # --- 1. Train Discriminator ---
            est_mag_det = est_mag.detach()
            with autocast_context(device, enabled=amp_enabled):
                pred_reals_d = _run_d(model_D, tgt_mag)
                pred_fakes_d = _run_d(model_D, est_mag_det)
                loss_D = adversarial_d_loss(pred_reals_d, pred_fakes_d)
            
            scaler_D.scale(loss_D / accum_steps).backward()
            parts['loss_D'] = loss_D.item()

            # --- 2. Train Generator ---
            for p in model_D.parameters():
                p.requires_grad_(False)
                
            use_fm = CONFIG['lambda_fm'] > 0
            with autocast_context(device, enabled=amp_enabled):
                if use_fm:
                    pred_fakes_g, fake_feats = _run_d(model_D, est_mag, return_features=True)
                    with torch.no_grad():
                        _, real_feats = _run_d(model_D, tgt_mag, return_features=True)
                    loss_fm_val = feature_matching_loss(real_feats, fake_feats)
                else:
                    pred_fakes_g = _run_d(model_D, est_mag)
                    loss_fm_val = 0.0
                loss_adv = adversarial_g_loss(pred_fakes_g)
                
            for p in model_D.parameters():
                p.requires_grad_(True)
                
            loss_G_total = loss_G_base + CONFIG['lamda_adv'] * loss_adv
            if use_fm:
                loss_G_total = loss_G_total + CONFIG['lambda_fm'] * loss_fm_val
                parts['loss_fm'] = float(loss_fm_val)
                
            scaler.scale(loss_G_total / accum_steps).backward()
            parts['loss_adv'] = loss_adv.item()
            parts['loss'] = loss_G_total.item()

            valid_accum_batches += 1
            if valid_accum_batches % accum_steps == 0:
                if CONFIG['grad_clip']:
                    scaler_D.unscale_(opt_D)
                    torch.nn.utils.clip_grad_norm_(model_D.parameters(), CONFIG['grad_clip'])
                scaler_D.step(opt_D)
                scaler_D.update()
                opt_D.zero_grad(set_to_none=True)
                
                if CONFIG['grad_clip']:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['grad_clip'])
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            train_losses.append(parts)
            pbar.set_postfix({'L_G': f"{parts['loss']:.3f}", 'L_D': f"{parts['loss_D']:.3f}"})

        avg_train = {k: float(np.mean([d[k] for d in train_losses if k in d])) for k in train_losses[0]}
        
        # Validation
        model.eval()
        model_D.eval()
        valid_losses = []
        with torch.no_grad():
            for batch in tqdm(valid_loader, desc=f'Epoch {epoch} [Valid]', leave=False):
                noisy = batch['noisy'].to(device, non_blocking=True)
                clean = batch['clean'].to(device, non_blocking=True)
                loss_G_base, parts, est_spec, tgt_spec = compute_loss_base(model, noisy, clean, CONFIG, window)
                
                est_mag = torch.sqrt(est_spec[..., 0]**2 + est_spec[..., 1]**2 + 1e-12).float()
                tgt_mag = torch.sqrt(tgt_spec[..., 0]**2 + tgt_spec[..., 1]**2 + 1e-12).float()
                
                with autocast_context(device, enabled=amp_enabled):
                    pred_reals_d = _run_d(model_D, tgt_mag)
                    pred_fakes_d = _run_d(model_D, est_mag)
                    loss_D = adversarial_d_loss(pred_reals_d, pred_fakes_d)
                    loss_adv = adversarial_g_loss(pred_fakes_d)
                    if use_fm:
                        _, fake_feats = _run_d(model_D, est_mag, return_features=True)
                        _, real_feats = _run_d(model_D, tgt_mag, return_features=True)
                        loss_fm_val = feature_matching_loss(real_feats, fake_feats)
                        parts['loss_fm'] = float(loss_fm_val)
                    
                loss_G_total = loss_G_base + CONFIG['lamda_adv'] * loss_adv
                if use_fm:
                    loss_G_total = loss_G_total + CONFIG['lambda_fm'] * loss_fm_val
                
                parts['loss_D'] = loss_D.item()
                parts['loss_adv'] = loss_adv.item()
                parts['loss'] = loss_G_total.item()
                valid_losses.append(parts)

        avg_valid = {k: float(np.mean([d[k] for d in valid_losses if k in d])) for k in valid_losses[0]}
        elapsed = time.time() - t0

        # Optional PESQ/STOI Eval
        pesq_stoi = {'pesq': 0.0, 'stoi': 0.0}
        eval_every = CONFIG.get('eval_pesq_every', 0)
        if eval_every > 0 and epoch % eval_every == 0:
            print(f'  Đang tính PESQ/STOI trên valid subset (mỗi {eval_every} epoch)...')
            pesq_stoi = compute_pesq_stoi(
                model, valid_loader, CONFIG, window, device, 
                max_samples=CONFIG.get('eval_pesq_samples', 50)
            )
            print(f'  >> PESQ: {pesq_stoi["pesq"]:.3f} | STOI: {pesq_stoi["stoi"]:.4f}')

        scheduler.step(avg_valid['loss'])
        scheduler_D.step(avg_valid['loss'])
        current_lr = optimizer.param_groups[0]['lr']

        is_best = best_loss is None or avg_valid['loss'] < best_loss
        if is_best: best_loss = avg_valid['loss']
        bad_epochs = 0 if is_best else bad_epochs + 1

        save_checkpoint(output_dir / 'last.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler, model_D, opt_D, scaler_D, scheduler_D)
        if is_best:
            save_checkpoint(output_dir / 'best.pt', model, optimizer, scheduler, epoch, best_loss, bad_epochs, scaler, model_D, opt_D, scaler_D, scheduler_D)

        print(
            f"Epoch {epoch:3d}/{CONFIG['epochs']} | "
            f"Train Loss: {avg_train['loss']:.4f} (D={avg_train['loss_D']:.4f}, adv={avg_train['loss_adv']:.4f}, fm={avg_train.get('loss_fm', 0.0):.4f}, ri={avg_train['ri_loss']:.4f}) | "
            f"Valid Loss: {avg_valid['loss']:.4f} (D={avg_valid['loss_D']:.4f}, adv={avg_valid['loss_adv']:.4f}, fm={avg_valid.get('loss_fm', 0.0):.4f}, ri={avg_valid['ri_loss']:.4f}) | "
            f"LR: {current_lr:.2e} | Time: {elapsed:.0f}s"
        )
        
        if wandb and wandb.run:
            wandb_metrics = {
                'train/loss_G': avg_train['loss'],
                'train/loss_D': avg_train['loss_D'],
                'train/loss_adv': avg_train['loss_adv'],
                'valid/loss_G': avg_valid['loss'],
                'valid/loss_D': avg_valid['loss_D'],
                'valid/loss_adv': avg_valid['loss_adv'],
                'lr': current_lr,
            }
            if use_fm:
                wandb_metrics['train/loss_fm'] = avg_train.get('loss_fm', 0.0)
                wandb_metrics['valid/loss_fm'] = avg_valid.get('loss_fm', 0.0)
            if pesq_stoi.get('pesq', 0) > 0:
                wandb_metrics['metrics/pesq'] = pesq_stoi['pesq']
                wandb_metrics['metrics/stoi'] = pesq_stoi['stoi']
                
            wandb.log(wandb_metrics, step=epoch)

else:
    print('run_training=False')



## 10. Kiểm tra nhanh sau training

Liệt kê checkpoint/log và nghe thử một mẫu từ validation set.


In [ ]:
import IPython.display as ipd

for name in ['best.pt', 'last.pt', 'config.json', 'notebook_config.json', 'train_log.csv']:
    p = output_dir / name
    if p.exists():
        print(f'{name}: {p} ({p.stat().st_size / 1024:.1f} KB)')
    else:
        print(f'MISSING: {p}')

if (output_dir / 'train_log.csv').exists():
    display(pd.read_csv(output_dir / 'train_log.csv').tail())

best_ckpt_path = output_dir / 'best.pt'
if best_ckpt_path.exists():
    print(f'Tải best checkpoint: {best_ckpt_path}')
    load_checkpoint(best_ckpt_path, model, device=device)

    model.eval()
    with torch.no_grad():
        batch = next(iter(valid_loader))
        noisy = batch['noisy'].to(device)
        clean = batch['clean'].to(device)
        spec = make_stft(noisy[:1], CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
        with autocast_context(device, enabled=use_amp):
            pred = model(spec)
        enhanced = make_istft(pred.float(), CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window, length=noisy[:1].shape[-1])

    sr = CONFIG['sample_rate']
    print('Noisy')
    display(ipd.Audio(noisy[0].detach().cpu().numpy(), rate=sr))
    print('Enhanced B3b')
    display(ipd.Audio(enhanced[0].detach().cpu().numpy(), rate=sr))
    print('Clean')
    display(ipd.Audio(clean[0].detach().cpu().numpy(), rate=sr))
else:
    print('Chưa có best.pt để nghe thử.')


## 11. Smoke test và Architecture Benchmark cho B3b

Chỉ chạy `B3b`, không đụng tới baseline/B1a/B1b/C1 checkpoint.


In [ ]:
import subprocess
import sys


def run_py(args, cwd=Path.cwd(), check=True):
    cmd = [sys.executable, *[str(arg) for arg in args]]
    print('\n$ ' + ' '.join(cmd), flush=True)
    result = subprocess.run(cmd, cwd=str(cwd), text=True)
    if check and result.returncode != 0:
        raise RuntimeError(f'Lệnh lỗi với exit code {result.returncode}: {cmd}')
    return result


AB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
arch_csv = Path(CONFIG['results_dir']) / 'ablation_arch_benchmark.csv'

if CONFIG.get('run_smoke_tests', True):
    run_py(['ablation/test_ablation_streaming.py', '--configs', CONFIG['config_id']])
else:
    print('run_smoke_tests=False, bỏ qua smoke test.')

if CONFIG.get('run_arch_benchmark', True):
    run_py([
        'ablation/run_ablation_benchmark.py',
        '--output', arch_csv,
        '--configs', CONFIG['config_id'],
        '--device', AB_DEVICE,
        '--frames', '63',
        '--warmup', '1',
        '--repeats', '3',
    ])
else:
    print('run_arch_benchmark=False, bỏ qua architecture benchmark.')

if arch_csv.exists():
    display(pd.read_csv(arch_csv))


## 12. Đánh giá chất lượng B3b: PESQ, STOI, SI-SDR, RTF
Đo lường các chỉ số chất lượng âm thanh sau khi đi qua mô hình:
- **PESQ (Perceptual Evaluation of Speech Quality)**: Đánh giá chất lượng giọng nói cảm nhận (điểm càng cao càng tốt, tối đa 4.5).
- **STOI (Short-Time Objective Intelligibility)**: Đánh giá độ rõ (hiểu được) của giọng nói (điểm từ 0 đến 1, càng cao càng tốt).
- **SI-SDR (Scale-Invariant Signal-to-Distortion Ratio)**: Đánh giá tỷ lệ tín hiệu trên nhiễu (điểm càng cao càng tốt).
- **RTF (Real-Time Factor)**: Tốc độ xử lý (thời gian xử lý / thời gian audio). RTF < 1 nghĩa là xử lý nhanh hơn thời gian thực.

Mục đích của block này là đảm bảo so sánh công bằng với baseline Colab V4a: cùng fixed `test.csv`, cùng cách load full-length audio, cùng STFT/ISTFT inference path, cùng metric implementation, cùng cách aggregate mean và cùng file CSV output. Nhờ vậy khác biệt giữa B3b và baseline đến từ mô hình/checkpoint, không đến từ pipeline đánh giá.

Cell này đánh giá trực tiếp `B3b_scratch_v1/best.pt` trên fixed `test.csv`, lưu bảng chi tiết/tổng hợp trong `output_dir/evaluation/`, đồng thời ghi `ablation_quality.csv` để cell tổng hợp kết quả phía sau vẫn chạy được.


In [ ]:
!pip install -q pesq pystoi torchmetrics pandas IPython tqdm

import time
import os
import pandas as pd
import numpy as np
import torch
import torchaudio
import warnings
from pesq import pesq
from pystoi import stoi
from torchmetrics.audio import ScaleInvariantSignalDistortionRatio
from IPython.display import display, FileLink
from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

quality_csv = Path(CONFIG['results_dir']) / 'ablation_quality.csv'
best_ckpt = output_dir / 'best.pt'


def metric_mean(df, column):
    values = pd.to_numeric(df[column], errors='coerce').to_numpy(dtype=float)
    finite = np.isfinite(values)
    return float(np.nanmean(values)) if finite.any() else float('nan')


def format_metric(value, digits=4):
    return 'nan' if np.isnan(value) else f'{value:.{digits}f}'


if CONFIG.get('run_quality_eval', True):
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy checkpoint để eval: {best_ckpt}')

    print(f'Tải trọng số tốt nhất từ: {best_ckpt}')
    load_checkpoint(best_ckpt, model, device=device)
    model.eval()

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Model params: {total_params:,} (trainable: {trainable_params:,})')
    print(f'Device: {device}')

    test_csv_path = CONFIG.get('test_csv', str(data_dir / 'test.csv'))
    print(f'Test CSV: {test_csv_path}')
    df_test = pd.read_csv(test_csv_path)

    # Đổi thành số lượng mẫu, ví dụ 50, nếu muốn chạy thử nhanh.
    EVAL_MAX_SAMPLES = None
    if EVAL_MAX_SAMPLES is not None:
        df_test = df_test.head(EVAL_MAX_SAMPLES)
    if df_test.empty:
        raise ValueError(f'Test CSV không có mẫu: {test_csv_path}')
    print(f'Test samples: {len(df_test)}')

    sisdr_metric = ScaleInvariantSignalDistortionRatio().to(device)

    results = []
    total_audio_duration = 0.0
    total_processing_time = 0.0
    EVAL_SR = CONFIG.get('sample_rate', 16000)

    with torch.no_grad():
        for idx, row in tqdm(df_test.iterrows(), total=len(df_test), desc='Evaluating B3b'):
            noisy_wav, sr_n = torchaudio.load(row['noisy_wav'])
            clean_wav, sr_c = torchaudio.load(row['clean_wav'])

            if noisy_wav.shape[0] > 1:
                noisy_wav = noisy_wav.mean(dim=0, keepdim=True)
            if clean_wav.shape[0] > 1:
                clean_wav = clean_wav.mean(dim=0, keepdim=True)

            if sr_n != EVAL_SR:
                noisy_wav = torchaudio.functional.resample(noisy_wav, sr_n, EVAL_SR)
            if sr_c != EVAL_SR:
                clean_wav = torchaudio.functional.resample(clean_wav, sr_c, EVAL_SR)

            noisy_wav = noisy_wav.squeeze(0)
            clean_wav = clean_wav.squeeze(0)
            min_audio_len = min(noisy_wav.numel(), clean_wav.numel())
            if min_audio_len <= 0:
                print(f'Bỏ qua mẫu rỗng: {row.get("ID", idx)}')
                continue
            noisy_wav = noisy_wav[:min_audio_len]
            clean_wav = clean_wav[:min_audio_len]

            audio_duration = noisy_wav.shape[0] / EVAL_SR
            total_audio_duration += audio_duration

            noisy_gpu = noisy_wav.unsqueeze(0).to(device)
            clean_gpu = clean_wav.unsqueeze(0).to(device)

            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_start = time.perf_counter()

            noisy_spec = make_stft(noisy_gpu, CONFIG['n_fft'], CONFIG['hop_length'], CONFIG['win_length'], window)
            amp_enabled = CONFIG.get('use_amp', False) and device.type == 'cuda'
            with autocast_context(device, enabled=amp_enabled):
                pred_stft = model(noisy_spec)

            pred_stft = pred_stft.float()
            enhanced = make_istft(
                pred_stft,
                CONFIG['n_fft'],
                CONFIG['hop_length'],
                CONFIG['win_length'],
                window,
                length=noisy_gpu.shape[-1],
            )

            if device.type == 'cuda':
                torch.cuda.synchronize()
            t_end = time.perf_counter()

            processing_time = t_end - t_start
            total_processing_time += processing_time
            rtf = processing_time / max(audio_duration, 1e-12)

            si_sdr_enhanced = sisdr_metric(enhanced, clean_gpu).item()
            si_sdr_noisy = sisdr_metric(noisy_gpu, clean_gpu).item()

            enhanced_np = enhanced.squeeze(0).detach().cpu().numpy()
            clean_np = clean_wav.numpy()
            noisy_np = noisy_wav.numpy()

            min_len = min(len(enhanced_np), len(clean_np), len(noisy_np))
            enhanced_np = enhanced_np[:min_len]
            clean_np = clean_np[:min_len]
            noisy_np = noisy_np[:min_len]

            try:
                pesq_enhanced = pesq(EVAL_SR, clean_np, enhanced_np, 'wb')
                pesq_noisy = pesq(EVAL_SR, clean_np, noisy_np, 'wb')
            except Exception:
                pesq_enhanced = float('nan')
                pesq_noisy = float('nan')

            try:
                stoi_enhanced = stoi(clean_np, enhanced_np, EVAL_SR, extended=False)
                stoi_noisy = stoi(clean_np, noisy_np, EVAL_SR, extended=False)
            except Exception:
                stoi_enhanced = float('nan')
                stoi_noisy = float('nan')

            results.append({
                'ID': row.get('ID', f'Sample_{idx}'),
                'PESQ_enhanced': round(pesq_enhanced, 4) if not np.isnan(pesq_enhanced) else float('nan'),
                'PESQ_noisy': round(pesq_noisy, 4) if not np.isnan(pesq_noisy) else float('nan'),
                'PESQ_improvement': round(pesq_enhanced - pesq_noisy, 4) if not (np.isnan(pesq_enhanced) or np.isnan(pesq_noisy)) else float('nan'),
                'STOI_enhanced': round(stoi_enhanced, 4) if not np.isnan(stoi_enhanced) else float('nan'),
                'STOI_noisy': round(stoi_noisy, 4) if not np.isnan(stoi_noisy) else float('nan'),
                'STOI_improvement': round(stoi_enhanced - stoi_noisy, 4) if not (np.isnan(stoi_enhanced) or np.isnan(stoi_noisy)) else float('nan'),
                'SI_SDR_enhanced_dB': round(si_sdr_enhanced, 2),
                'SI_SDR_noisy_dB': round(si_sdr_noisy, 2),
                'SI_SDR_improvement_dB': round(si_sdr_enhanced - si_sdr_noisy, 2),
                'RTF': round(rtf, 6),
                'duration_s': round(audio_duration, 3),
            })

    if not results:
        raise RuntimeError('Không đánh giá được mẫu nào trong test set.')

    df_results = pd.DataFrame(results)

    pesq_enh = metric_mean(df_results, 'PESQ_enhanced')
    pesq_noisy = metric_mean(df_results, 'PESQ_noisy')
    pesq_delta = metric_mean(df_results, 'PESQ_improvement')
    stoi_enh = metric_mean(df_results, 'STOI_enhanced')
    stoi_noisy = metric_mean(df_results, 'STOI_noisy')
    stoi_delta = metric_mean(df_results, 'STOI_improvement')
    sisdr_enh = metric_mean(df_results, 'SI_SDR_enhanced_dB')
    sisdr_noisy = metric_mean(df_results, 'SI_SDR_noisy_dB')
    sisdr_delta = metric_mean(df_results, 'SI_SDR_improvement_dB')
    rtf_mean = metric_mean(df_results, 'RTF')
    overall_rtf = total_processing_time / max(total_audio_duration, 1e-12)

    df_summary = pd.DataFrame({
        'Metric': [
            'PESQ (enhanced)', 'PESQ (noisy)', 'PESQ (Δ improvement)',
            'STOI (enhanced)', 'STOI (noisy)', 'STOI (Δ improvement)',
            'SI-SDR enhanced (dB)', 'SI-SDR noisy (dB)', 'SI-SDR Δ (dB)',
            'RTF (mean)', 'RTF (overall)', 'Real-time capable?',
        ],
        'Value': [
            format_metric(pesq_enh, 4),
            format_metric(pesq_noisy, 4),
            format_metric(pesq_delta, 4),
            format_metric(stoi_enh, 4),
            format_metric(stoi_noisy, 4),
            format_metric(stoi_delta, 4),
            format_metric(sisdr_enh, 2),
            format_metric(sisdr_noisy, 2),
            format_metric(sisdr_delta, 2),
            format_metric(rtf_mean, 6),
            format_metric(overall_rtf, 6),
            'Có' if overall_rtf < 1.0 else 'Không',
        ],
    })

    print('\n' + '=' * 60)
    print('  KẾT QUẢ ĐÁNH GIÁ CHẤT LƯỢNG MÔ HÌNH DeepVQE B3b')
    print('=' * 60)
    print(f'Test samples: {len(df_results)}')
    print(f'Model params: {total_params:,}')
    print(f'Device: {device}')
    print(f'Total audio: {total_audio_duration:.1f}s | Processing: {total_processing_time:.1f}s')
    print()
    display(df_summary)

    print('\n--- Chi tiết 10 mẫu đầu tiên ---')
    display(df_results.head(10))

    save_dir = output_dir / 'evaluation'
    save_dir.mkdir(parents=True, exist_ok=True)

    detail_csv = save_dir / 'eval_metrics_per_sample.csv'
    summary_csv_path = save_dir / 'eval_metrics_summary.csv'

    df_results.to_csv(detail_csv, index=False)
    df_summary.to_csv(summary_csv_path, index=False)

    print(f'\nĐã lưu kết quả chi tiết:  {detail_csv}')
    print(f'Đã lưu bảng tổng hợp:     {summary_csv_path}')
    display(FileLink(str(detail_csv)))
    display(FileLink(str(summary_csv_path)))

    quality_csv.parent.mkdir(parents=True, exist_ok=True)
    metadata = reproducibility_metadata(TRAIN_CFG, checkpoint_id=best_ckpt.stem)
    quality_row = {
        'config_id': CONFIG['config_id'],
        **metadata,
        'checkpoint_id': best_ckpt.stem,
        'num_eval_items': len(df_results),
        'eval_duration_s': total_audio_duration,
        'pesq': pesq_enh,
        'stoi': stoi_enh,
        'si_sdr': sisdr_enh,
        'dnsmos_ovrl': '',
        'dnsmos_sig': '',
        'dnsmos_bak': '',
        'erle': '',
        'notes': 'notebook quality eval with PESQ/STOI/SI-SDR/RTF; noisy baselines saved in evaluation detail CSV',
    }
    quality_fields = [
        'config_id',
        'git_commit',
        'config_hash',
        'seed',
        'hardware_info',
        'torch_version',
        'onnxruntime_version',
        'num_threads',
        'checkpoint_id',
        'num_eval_items',
        'eval_duration_s',
        'pesq',
        'stoi',
        'si_sdr',
        'dnsmos_ovrl',
        'dnsmos_sig',
        'dnsmos_bak',
        'erle',
        'notes',
    ]
    if quality_csv.exists():
        existing_quality = pd.read_csv(quality_csv, dtype=str).to_dict('records')
    else:
        existing_quality = []
    existing_quality = [row for row in existing_quality if row.get('config_id') != CONFIG['config_id']]
    existing_quality.append({field: quality_row.get(field, '') for field in quality_fields})
    pd.DataFrame(existing_quality, columns=quality_fields).to_csv(quality_csv, index=False)
    print(f'Đã cập nhật ablation quality CSV: {quality_csv}')
    display(pd.read_csv(quality_csv))
else:
    print('run_quality_eval=False, bỏ qua eval.')
    if quality_csv.exists():
        display(pd.read_csv(quality_csv))


## 13. ONNX export/parity cho B3b (tuỳ chọn)


In [ ]:
onnx_csv = Path(CONFIG['results_dir']) / 'ablation_onnx.csv'
onnx_dir = Path(CONFIG['onnx_dir'])

if CONFIG.get('run_onnx_export', False):
    if not best_ckpt.exists():
        raise FileNotFoundError(f'Không tìm thấy checkpoint để export ONNX: {best_ckpt}')
    run_py([
        'ablation/export_ablation_onnx.py',
        '--config-id', CONFIG['config_id'],
        '--checkpoint', best_ckpt,
        '--output-dir', onnx_dir,
        '--results', onnx_csv,
        '--device', 'cpu',
    ])
else:
    print('run_onnx_export=False, bỏ qua ONNX export.')

if onnx_csv.exists():
    display(pd.read_csv(onnx_csv))


## 14. Tổng hợp CSV và nén kết quả

Tạo `ablation_summary.csv` cho riêng B3b và zip toàn bộ workspace output để tải từ Colab.


In [ ]:
from IPython.display import FileLink, display

summary_csv = Path(CONFIG['results_dir']) / 'ablation_summary.csv'
run_py([
    'ablation/collect_ablation_results.py',
    '--arch', arch_csv,
    '--quality', quality_csv,
    '--onnx', onnx_csv,
    '--output', summary_csv,
])

print('\nCSV kết quả:')
for csv_path in [arch_csv, quality_csv, onnx_csv, summary_csv]:
    if csv_path.exists():
        print(f'  {csv_path} ({csv_path.stat().st_size / 1024:.1f} KB)')
        display(pd.read_csv(csv_path))

archive_base = Path('/content/b3b_ablation_training_output')
archive_path = archive_base.with_suffix('.zip')
if archive_path.exists():
    archive_path.unlink()
shutil.make_archive(str(archive_base), 'zip', root_dir=str(WORK_DIR))
print(f'Đã nén kết quả: {archive_path}')
display(FileLink(str(archive_path)))
